# 📰 Fake News Detection using Graph ML
## Notebook 1: Data Collection & Exploration

**Project Pipeline:**
```
01_data_exploration ➜ 02_graph_construction ➜ 03_gnn_training ➜ 04_visualization
```

**Dataset:** LIAR Dataset — 12,836 labeled short statements from PolitiFact  
**Labels:** We simplify 6 classes ➜ Binary (Real / Fake)

## Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries
!pip install datasets transformers scikit-learn matplotlib seaborn wordcloud -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ Libraries loaded successfully!')

## Step 2 — Download the LIAR Dataset

In [ ]:
from datasets import load_dataset

# Load LIAR dataset from HuggingFace Hub
dataset = load_dataset('liar')
print('✅ Dataset loaded!')
print(dataset)

In [ ]:
# Convert to pandas DataFrames
train_df = pd.DataFrame(dataset['train'])
val_df   = pd.DataFrame(dataset['validation'])
test_df  = pd.DataFrame(dataset['test'])

print(f'Train size : {len(train_df):,}')
print(f'Val size   : {len(val_df):,}')
print(f'Test size  : {len(test_df):,}')
print(f'\nColumns: {list(train_df.columns)}')
train_df.head()

## Step 3 — Preprocess: 6 Labels ➜ Binary (Real / Fake)

In [ ]:
# LIAR has 6 labels: pants-fire(0), false(1), barely-true(2), half-true(3), mostly-true(4), true(5)
# We binarise: 0,1,2 = FAKE (1)  |  3,4,5 = REAL (0)

LABEL_MAP = {0: 1, 1: 1, 2: 1,   # Fake
             3: 0, 4: 0, 5: 0}    # Real

LABEL_NAMES = {0: 'pants-fire', 1: 'false', 2: 'barely-true',
               3: 'half-true',  4: 'mostly-true', 5: 'true'}

for df in [train_df, val_df, test_df]:
    df['binary_label']  = df['label'].map(LABEL_MAP)
    df['label_name']    = df['label'].map(LABEL_NAMES)
    df['statement']     = df['statement'].astype(str).str.strip()
    df['speaker']       = df['speaker'].astype(str).str.strip()
    df['subject']       = df['subject'].astype(str).str.strip()
    df['state_info']    = df['state_info'].astype(str).str.strip()
    df['party_affiliation'] = df['party_affiliation'].astype(str).str.strip()

print('Original label distribution (train):')
print(train_df['label_name'].value_counts())
print(f'\nBinary — Fake: {(train_df.binary_label==1).sum():,}  |  Real: {(train_df.binary_label==0).sum():,}')

## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Original 6-class distribution
label_counts = train_df['label_name'].value_counts()
colors_6 = ['#e74c3c','#e67e22','#f39c12','#27ae60','#2980b9','#8e44ad']
axes[0].bar(label_counts.index, label_counts.values, color=colors_6)
axes[0].set_title('Original 6-class label distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Plot 2: Binary distribution
binary_counts = train_df['binary_label'].value_counts()
axes[1].pie(binary_counts.values,
            labels=['Fake', 'Real'],
            colors=['#e74c3c', '#2ecc71'],
            autopct='%1.1f%%',
            startangle=90,
            explode=(0.05, 0.05))
axes[1].set_title('Binary label distribution (train)')

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot saved as label_distribution.png')

In [ ]:
# Statement length analysis
train_df['stmt_len'] = train_df['statement'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Length distribution by label
fake_lens = train_df[train_df.binary_label==1]['stmt_len']
real_lens = train_df[train_df.binary_label==0]['stmt_len']
axes[0].hist(fake_lens, bins=30, alpha=0.6, color='#e74c3c', label='Fake', density=True)
axes[0].hist(real_lens, bins=30, alpha=0.6, color='#2ecc71', label='Real', density=True)
axes[0].set_title('Statement length distribution')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Density')
axes[0].legend()

# Top speakers
top_speakers = train_df['speaker'].value_counts().head(10)
axes[1].barh(top_speakers.index[::-1], top_speakers.values[::-1], color='#3498db')
axes[1].set_title('Top 10 speakers')
axes[1].set_xlabel('Statement count')

plt.tight_layout()
plt.savefig('statement_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Word clouds: Fake vs Real
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label_val, title, color in zip(
        axes, [1, 0],
        ['FAKE news word cloud', 'REAL news word cloud'],
        ['Reds', 'Greens']):
    text = ' '.join(train_df[train_df.binary_label == label_val]['statement'])
    wc = WordCloud(width=700, height=400,
                   background_color='white',
                   colormap=color,
                   max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top subjects
top_subjects = train_df['subject'].value_counts().head(15)
plt.figure(figsize=(12, 5))
plt.bar(top_subjects.index, top_subjects.values, color='#9b59b6')
plt.title('Top 15 news subjects')
plt.xlabel('Subject')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('subjects.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Save Processed Data

In [ ]:
import os
os.makedirs('data', exist_ok=True)

train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv', index=False)
test_df.to_csv('data/test.csv', index=False)

print('✅ Processed data saved to data/ folder')
print(f'   Train: {len(train_df):,} rows')
print(f'   Val  : {len(val_df):,} rows')
print(f'   Test : {len(test_df):,} rows')
print('\n🚀 Ready for Notebook 2: Graph Construction!')